In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.metrics import mean_squared_error

from lightgbm import LGBMRegressor

# =========================================================
# LOAD DATA
# =========================================================

train = pd.read_csv("cleaned_train.csv")
test = pd.read_csv("cleaned_test.csv")

# =========================================================
# TARGET + FEATURES
# =========================================================

y = train['SalePrice']

X = train.drop(['SalePrice', 'Id'], axis=1)

X_test = test.drop(['Id'], axis=1)

test_ids = test['Id']

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================

X.columns = X.columns.str.replace(
    '[^A-Za-z0-9_]+',
    '',
    regex=True
)

X_test.columns = X_test.columns.str.replace(
    '[^A-Za-z0-9_]+',
    '',
    regex=True
)

# =========================================================
# ALIGN TRAIN + TEST COLUMNS
# =========================================================

X, X_test = X.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

# =========================================================
# TRAIN VALID SPLIT
# =========================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# LIGHTGBM MODEL
# =========================================================

model = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=10,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42
)

# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(X_train, y_train)

# =========================================================
# VALIDATION PREDICTION
# =========================================================

y_pred = model.predict(X_valid)

# =========================================================
# RMSE SCORE
# =========================================================

rmse = np.sqrt(
    mean_squared_error(y_valid, y_pred)
)

print("Validation RMSE:", rmse)

# =========================================================
# CROSS VALIDATION SCORE
# =========================================================

cv_scores = np.sqrt(
    -cross_val_score(
        model,
        X,
        y,
        scoring='neg_mean_squared_error',
        cv=5
    )
)

print("Cross Validation RMSE:", cv_scores.mean())

# =========================================================
# TRAIN ON FULL DATA
# =========================================================

model.fit(X, y)

# =========================================================
# TEST PREDICTIONS
# =========================================================

# =========================================================
# TEST PREDICTIONS
# =========================================================

predictions = model.predict(X_test)

# convert back from log scale
predictions = np.expm1(predictions)

# =========================================================
# CREATE SUBMISSION FILE
# =========================================================

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5943
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 152
[LightGBM] [Info] Start training from score 12.030658
Validation RMSE: 0.13894422205006024
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5924
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 155
[LightGBM] [Info] Start training from score 12.021409
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001822 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5931
[LightGBM] [Info] Number of data points in the train set: 1168, number of used fe

In [ ]:
train['SalePrice'].head()

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64